# Assignment 2: Exploring sources of bias in data
## Bias in data, demystified

**All datasets have limitations. Any of these limitations can be a potential source of bias.** In the context of data, a "bias" is really just a divergence between what a dataset is supposed to capture about the world--according to the *dataset designer's intention*, or according to the *end-user's expectations*--and what's actually represented in the data. 

So biases in data are often highly *contextual*, which can make them subtle and hard to spot. Similarly, it's hard to predict ahead of time what the *consequences* of those biases might be, because it depends on what the data is being used for.

Nevertheless some sources of "bias" in data show up over and over again. Some of these are:
- **duplicated data:** the same data shows up multiple times
- **incomplete data:** some data is missing from the dataset
- **misleading data:** it looks like a piece of data means one thing, but it actually means something different
- **unrepresentative data:** the dataset doesn't represent the population it's gathered from OR the population it's intended to model


Any of these sources of bias, unless their properly documented (with a data statement, etc) can have unintended consequences: they cause a researcher who is using the data to reach incorrect conclusions, or cause a machine learning model that is trained on that dataset to make classification errors.

The nature or scale of these consequences can be hard to predict. That's why any time you prepare to use data that you didn't gather yourself, it pays to spend some time exploring the dataset, identifying limitations, and thinking critically about how these limitations might affect your analysis or your machine learning model.

## Introduction to Assignment 2
For this assignment, you'll be working with the Wikipedia Talk Corpus. You have already created a data statement for this dataset in class, so you're an 'expert' on it compared to most people!

**In assignment 1, you learned how to process and analyze a dataset created by someone else.** In that assignment, we took the dataset at face value: we assumed that there were no major errors, no missing data, nothing that would skew our results. We assumed that the dataset really did accurately capture all the bike and pedestrian traffic on the Burke-Gilman trail over the specified period of time.

**In assignment 2, we'll also be analyzing a dataset created by someone else,** but this time *we won't assume that the data is complete and correct.* Instead, we will try to identify ways in which the data might be WRONG, and form hypotheses about how the limitations we discover might make it an unsuitable, or at least a potentially problematic, training dataset for a general-purpose hostile speech detector. 

**There are 16 questions in this assignment, across three sections:**

***In part 1 of this assignment,*** you will load one type of Wikipedia Talk Corpus data--the demographics of the crowdworkers who labelled the comments--into your copy of this Jupyter Notebook, and we will walk through a series of data processing steps, with the goal of ending up with a complete and accurate set of data about all of those crowdworkers. In the process, we will discover and discuss several limitations of the dataset.

***In part 2 of this assignment,*** you will generate some basic descriptive statistics about the demographics of the crowdworkers described in that dataset. If you are comfortable in Python, you can perform that analysis in this Notebook. If you aren't as comfortable in Python (yet!), you can perform the analysis in Google Sheets (just be sure to link to that Google Sheet from this notebook, and set the permissions so that your instructors can view it!). 

***In part 3 of this assignment,*** you will answer some additional research questions about this dataset. Some of these questions can be answered without writing additional code or analyzing additional data; other questions will require you to combine this dataset another dataset in the corpus. You will have the option to choose whether you want to answer code or no-code questions.

Whether you choose code or no-code question for part 3, you will need to write your responses to the question within this notebook, and submit a link to this notebook for grading.

Part 3 also contains some optional "challenge" questions that you can investigate if you want to (hint: any one of these challenge questions could be the focus of a final course project).


***Note:*** *Since the purpose of this class is to get you comfortable thinking critically (like a researcher), rather than programming perfectly, you won't be graded on your code. You'll be graded on how well you DOCUMENT YOUR PROCESS and REFLECT on the implications of your findings. If you still aren't sure what that means, ask your instructor or TA!*

## Part 1: Cleaning and analyzing the annotator demographic data

For the first part of this assignment, we're going to prepare one set of Wikipedia Talk data--the annotator demographics files--for analysis. In the process we'll perform a few "sanity checks" to make sure we understand what the data means, and know any limitations.

This sort of "[data wrangling](https://en.wikipedia.org/wiki/Data_wrangling)" is a critical, if sometimes tedious, first step for any quantitative research project.

### 1.1 Load the data into the notebook

According to the documetation, the worker demographic data for the Wikipedia Talk Corpus is spread across three files:
- ``toxicity_worker_demographics.tsv``
- ``aggression_worker_demographics.tsv``
- ``attack_worker_demographics.tsv``

We will need to combine the data in these three files to come up with our canonical list of workers. 
**Just like working with data in the real world, you will need to retrieve the files from their source (Figshare) and transfer them into your notebook server**.

- **Toxicity dataset:** [data](https://figshare.com/articles/dataset/Wikipedia_Talk_Labels_Toxicity/4563973) (Note: this is 35Mb!), [documentation](https://meta.wikimedia.org/wiki/Research:Detox/Data_Release#Toxicity), [questionnaire shown to crowdworkers](https://github.com/ewulczyn/wiki-detox/blob/master/src/modeling/toxicity_question.png)
- **Aggression dataset:** [data](https://figshare.com/articles/dataset/Wikipedia_Talk_Labels_Aggression/4267550) (Note: this is 74Mb!), [documentation](https://meta.wikimedia.org/wiki/Research:Detox/Data_Release#Aggression), [questionnaire shown to crowdworkers](https://github.com/ewulczyn/wiki-detox/blob/master/src/modeling/aggression_question.png)
- **Personal attacks dataset:** [data](https://figshare.com/articles/dataset/Wikipedia_Talk_Labels_Personal_Attacks/4054689) (this is only 92k), [documentation](https://meta.wikimedia.org/wiki/Research:Detox/Data_Release#Personal_Attacksv), [questionnaire shown to crowdworkers](https://github.com/ewulczyn/wiki-detox/blob/master/src/modeling/attack_question.png)

First we'll load each of the annotator datafiles into our Notebook and save them into data structures that's easy to work with. In this case, I'm choosing to save each of these files as a list-of-dictionaries, since that's fairly standard, and it makes it easy to check your work as you go.

By the way: ``.tsv`` stands for "tab-separated values", and it means that this file is organized into rows and columns, like a spreadsheet, and the data values for each column are separated by "tab" characters.

In [2]:
#import the csv module, a little code toolkit for working with spreadsheet-style data files
import csv 

The function below will load in a tab-separated (.tsv) file and convert it into a lists-of-dictionaries. 

If you don't have much experience with Python, this (and some of the other code in this notebook) might be hard to understand. That's okay! For now, it's most important that you know what it does.

If you have a ***lot*** of experience with Python, the code in this notebook might seem really, really primitive. That's also okay! Remember: in this course we're primarily interested in data, not code. Code is just one of the many tools we use to ask and answer questions about data.

In [3]:
def prepare_datasets(file_path):
    """ 
    Accepts: path to a tab-separated plaintext file
    Returns: a list containing a dictionary for every row in the file, 
        with the file column headers as keys
    """
    
    with open(file_path) as infile:
        reader = csv.DictReader(infile, delimiter='\t')
        list_of_dicts = [dict(r) for r in reader]
        
    return list_of_dicts

### 1.2 Identifying duplicated datafiles
Let's load our three .tsv files into Python and store them as three variables with relevant names, so that we know which is which. 

Once we've created these three lists-of-dicts, we will do two things to check our work so far: 
- we will print the first annotator's demographic data (list index ``[0]``) so that we know what the format looks like
- we will print the length of each list (the ``len`` function), to see how many rows is in each file. Each row should correspond to one crowdworker/annotator.

***Note:*** for the cell below to run, your version of these datafiles and folders will need have the same names as the ones below, and your version of this Notebook will need to be stored in the same directory as the three folders that hold the datafiles.

In [4]:
#load the data from the flat files into three lists-of-dictionaries
toxicity_annotators = prepare_datasets("wikipedia_datasets/toxicity_worker_demographics.tsv")
print(toxicity_annotators[0])
print(len(toxicity_annotators))

attack_annotators = prepare_datasets("wikipedia_datasets/attack_worker_demographics.tsv")
print(attack_annotators[0])
print(len(attack_annotators))

aggression_annotators = prepare_datasets("wikipedia_datasets/aggression_worker_demographics.tsv")
print(aggression_annotators[0])
print(len(aggression_annotators))



{'worker_id': '85', 'gender': 'female', 'english_first_language': '0', 'age_group': '18-30', 'education': 'bachelors'}
3591
{'worker_id': '833', 'gender': 'female', 'english_first_language': '0', 'age_group': '45-60', 'education': 'bachelors'}
2190
{'worker_id': '833', 'gender': 'female', 'english_first_language': '0', 'age_group': '45-60', 'education': 'bachelors'}
2190


Interesting! This tells us a few things that we didn't know before:
1. it looks like the demographic data matches what's listed in [the schema](https://meta.wikimedia.org/wiki/Research:Detox/Data_Release#Schema_for_{attack/aggression/toxicity}_worker_demographics.tsv), which is great!
2. it looks like the "toxicity" dataset was annotated by a lot more people (3,591) than the "attack" or "aggression" datasets (2,190)
3. ``aggression_worker_demographics.tsv`` and ``attack_worker_demographics.tsv`` seem to contain the same number of workers, and the worker at the beginning of each list has the same ID and demographic data.

Let's dig a little deeper into finding #3. Are the ***last*** entries in both of these lists also identical?

In [5]:
print(attack_annotators[-1]) # "-1" tells Python to find the last item in any list
print(aggression_annotators[-1])


{'worker_id': '3876', 'gender': 'female', 'english_first_language': '1', 'age_group': '30-45', 'education': 'bachelors'}
{'worker_id': '3876', 'gender': 'female', 'english_first_language': '1', 'age_group': '30-45', 'education': 'bachelors'}


Yes, the last rows in these two lists are also identical! 

And in fact if you had opened the two lists in a text editor or spreadsheet program, you would find that the *aggression and attack .tsv files contain exactly the same data.* By the way, it doesn't say anywhere in the dataset documentation that these two files are identical!

That brings us to our first lesson about bias: watch out for duplicate data! 

Consider: *What would have happened if we had just combined these three files and then analyzed the worker demographics? What mistaken conclusions might we have drawn from that?*
 
Fortunately, now that we know that there is duplicate data we can work around it. Since two files are identical, we only need to use one of them. So from now on, we will ignore ``aggression_annotators`` entirely. 

Since want to remember that ``attack_annotators`` really refers to both "attack" and "aggression" annotators, we can just rename the variable we're using to store that dataset.

In [6]:
attack_aggro_annotators = attack_annotators

Okay, that looks good. Now we can continue getting our data ready for analysis--while keeping an eye out for additional duplicate data and other "gotchas"!

### 1.3 Understanding what the properties of your data really mean

Whenever you are working with data that you didn't create, it's very useful to perform some basic sanity-checking to make sure the data actually means what you think it means. 

For example, take the ``worker_id`` field in our datasets. 

The [schema](https://meta.wikimedia.org/wiki/Research:Detox/Data_Release#Schema_for_{attack/aggression/toxicity}_worker_demographics.tsv) says that the ``worker_id`` field contains an "anonymized crowd-worker id" and that this ID is meant to join the worker demographics datafiles with the annotator comments datafiles, so that if we wanted find all of the comments that worker "85" (from above) labelled, we could find her by looking for that ID in each row of ``toxicity_annotated_comments.tsv``.

So far, so good. But since we want to combine the toxicity and attack + aggression annotator demographics data into a single dataset, we probably want to know...

1. did any of the values for ``worker_id`` appear in both datasets?
2. if so, do they correspond to the same person (or at least, a person with matching gender, age group, etc.)?

### 1.4 Checking for duplicate worker IDs

If we want to see if worker_id means the same thing across datasets, the first step is to see if any of the same worker_ids exist across the two datasets.

To check this, let's first pull all the worker IDs out of each dataset and combine them into a single list. Then we can check that list to see if it contains any duplicate values. If it does, we know that there is at least 1 value for ``worker_id`` that appears in both datasets.

**Note:** *We're assuming that there are no duplicate values for ``worker_id`` within each dataset. (There aren't, I checked). It's a pretty safe assumption though, because worker_id is intended to be a unique key that links the ``worker_demographics.tsv`` files and the ``annotated_comments.tsv`` datasets. Can you explain why it would be an issue if there were duplicate values for ``worker_id`` within an individual dataset?*

In [28]:
#pull the worker ids out of the individual files
tox_w_ids = [item['worker_id'] for item in toxicity_annotators]
aa_w_ids = [item['worker_id'] for item in attack_aggro_annotators]


#create a new list to hold all the ids
all_w_ids = tox_w_ids + aa_w_ids

# #combine the two worker_id lists into the new list
# all_w_ids.extend(tox_w_ids)
# all_w_ids.extend(aa_w_ids)

In [29]:
#how many worker ids do we have, total? 
#this number should match the total count of toxicity_annotators and attack_aggro_annotators (3591 + 2190 = 5781)
print(len(all_w_ids))

5781


In [30]:
#what does our new list look like? Let's print the first five values
print(all_w_ids[0:5])

['85', '1617', '1394', '311', '1980']


Below is our duplicate-checker function. You pass it a list of values, and it will return "True" if it finds at least 1 duplicate value in that list. Can you figure out how it works?

In [31]:
def determine_dupes(w_ids):
    set_of_ids = set()
    
    found_a_dupe = False
    
    for w in w_ids:
        if w in set_of_ids:
            found_a_dupe = True
        else:
            set_of_ids.add(w)

    return found_a_dupe

In [32]:
#we'll call determine_dupes to check for duplicates in the combined worker id list
has_dupes = determine_dupes(all_w_ids)

#if this prints 'True', that means we found at least one duplicate ID
print(has_dupes)

True


Hmmm... So it looks like there's at least 1 duplicate! Okay, we'll need to do some additional verification before we can decide what to do with that information. 

The next thing we'll do is check how many duplicates there are. We'll write a short script that reads through ``all_w_ids`` and every time it finds a value that appears more than once, it adds that value to a new list.

Can you figure out how the script below works?

In [33]:
#create an empty list to hold any duplicate worker_id values we find
dupes = []

#look through the data, if you encounter any value more than once, add it to our 'dupes' list
for w in all_w_ids:
    if all_w_ids.count(w) > 1:
        dupes.append(w)

#how many values were added to 'dupes'?        
print(len(dupes))

#how many worker_ids are present twice in the dataset?
#can you explain why we are dividing the length of "dupes" by 2 in order to answer that question?
print(len(dupes)/2)

3716
1858.0


Huh, so it looks like 1,858 of the ``worker_ids`` in our merged dataset are duplicates! That's a lot of duplication, since our list was only 5,781 rows in the first place, including these dupes! 

We will definitely need to account for these duplicates before we start analyzing worker demographics. 
- If the duplicate ``worker_id``s have different demographic metadata, we will assume they are different people with the same ID. 
- If the metadata is identical, then we know we can safely remove the duplicates
- If some metadata is identical and some is different... well, let's hope that's not the case!

### 1.5 Check for duplicate worker demographic metadata
Let's see if it's just the value for ``worker_id`` that is duplicated across the two datasets (meaning that the these duplicate ids correspond to different workers with different demographics), or if ``worker_id`` really corresponds to the same people across ``toxicity_annotators`` and ``attack_annotators``.

To do this we'll perform some spot checks, meaning we'll visually compare the demographic data of workers with duplicate IDs, to see if they look like the same worker, or not. Running random 'spot checks' is a really common and useful way of finding systematic issues with your data without having to check every entry. 
 

First, we'll extract 10 random ids from our dupe set. Using a random sample (rather than just looking at the first 10 rows, for instance) helps us be more confident that any patterns we see are really there.

In [34]:
#handy Python library that lets you select things randomly from a list
import random

In [35]:
#convert our 'dupes' list into a set
#bonus: can you explain why we created "dupeset" rather than just grabbing 10 random values from "dupes"?
#hint: in Python, a 'set' is like a list that can only contain a single instance of any value
dupeset = sorted(set(dupes))

#store our random sample of dupes in its own list
dupe_sample = random.sample(dupeset, 10)

#print to confirm everything looks how we expect it to...
print(dupe_sample)

['2640', '1931', '105', '3779', '3354', '1944', '2634', '1103', '1023', '2951']


Now, we'll use this ``dupe_sample`` list that we created to pull the corresponding worker demographics from each of our two datasets, using the function below. See if you can figure out how the function works!

In [36]:
def worker_id_lookup(annotator_list, dupe_id):
    """
    Accepts: a list of dictionaries 
        & a list of known duplicate values for
        the key 'worker_id' in those dictionaries
    
    If a duplicate value for worker_id is found, 
        print the complete dictionary of demographic data for that worker
    
    """
    
    for a in annotator_list:
        if a['worker_id'] == dupe_id:
            print(a)
            
#loop through the duplicate sample list and call our worker_id_lookup function
#to check each dataset for corresponding worker demographic data
for d in dupe_sample:
    worker_id_lookup(toxicity_annotators, d)
    worker_id_lookup(attack_aggro_annotators, d)

{'worker_id': '2640', 'gender': 'male', 'english_first_language': '0', 'age_group': '18-30', 'education': 'professional', 'dataset': 'toxicity', 'unique_id': '2507'}
{'worker_id': '2640', 'gender': 'male', 'english_first_language': '1', 'age_group': '30-45', 'education': 'bachelors', 'dataset': 'attack and aggression', 'unique_id': '4683'}
{'worker_id': '1931', 'gender': 'female', 'english_first_language': '1', 'age_group': '45-60', 'education': 'hs', 'dataset': 'toxicity', 'unique_id': '398'}
{'worker_id': '1931', 'gender': 'male', 'english_first_language': '0', 'age_group': '30-45', 'education': 'bachelors', 'dataset': 'attack and aggression', 'unique_id': '4495'}
{'worker_id': '105', 'gender': 'female', 'english_first_language': '0', 'age_group': '30-45', 'education': 'hs', 'dataset': 'toxicity', 'unique_id': '77'}
{'worker_id': '105', 'gender': 'female', 'english_first_language': '0', 'age_group': '30-45', 'education': 'masters', 'dataset': 'attack and aggression', 'unique_id': '47

Aha! It looks like few if any of these ``worker_id`` values correspond to the same demographic data info across the two datasets. 

It's possible that some of the same people labelled both datasets and were assigned different IDs each time. But there's no way for us to determine that with the data we have. 

So for now we will assume that these datasets were labelled by two entirely different sets of crowdworkers. Which means we can combine these two datasets into one, with no duplication to mess up our analysis.

### 1.6 Final preparation of the dataset

Before we start our analysis of worker demographics, let's do two more things:

1. Since we know now that worker ID isn't unique, let's give each worker in our new, combined dataset a **truly** unique ID.
2. While we're at it, let's also add a new field to each worker's demographic dictionary that lists which dataset the worker worked on (toxicity or attack/aggressive).

In [37]:
def add_dataset_id(list_of_dicts, dataset_ref):
    """
    Accepts: a list of dictionaries & a strig value for "dataset"
    Returns: that list of dictionaries, with the new "dataset" key and the specified value
    """
        
    for w in list_of_dicts:
        w.update({"dataset" : dataset_ref})
        
    return list_of_dicts

In [38]:
#update our worker demographic datasets with the new key and value
toxicity_annotators = add_dataset_id(toxicity_annotators, "toxicity")
attack_aggro_annotators = add_dataset_id(attack_aggro_annotators, "attack and aggression")

In [39]:
#did it work?
print(toxicity_annotators[0])
print(attack_aggro_annotators[0])

{'worker_id': '85', 'gender': 'female', 'english_first_language': '0', 'age_group': '18-30', 'education': 'bachelors', 'dataset': 'toxicity', 'unique_id': '1'}
{'worker_id': '833', 'gender': 'female', 'english_first_language': '0', 'age_group': '45-60', 'education': 'bachelors', 'dataset': 'attack and aggression', 'unique_id': '3592'}


Great. Now we'll use the function below to... 

1. combine the two datasets into one
2. assign a truly unique id to each worker 

It doesn't matter what this ID is, as long as its unique within the dataset. We'll use sequential IDs, starting at "1" and going up from there.

In [40]:
def finalize_dataset(list1, list2):
    
    #combine the two lists into a new one
    combined_list = list1 + list2
    
    #initialize our counter variable
    new_id = 1
    
    for w in combined_list:
        #add the new sequential id field, and populate with the current value of 'counter'
        w.update({"unique_id" : str(new_id)})
        
        #increment the counter variable by 1, so that the next ID will be one number higher
        new_id = new_id + 1
        
    return combined_list

In [43]:
#create our new combined list with sequential IDs
all_annotators = finalize_dataset(toxicity_annotators, attack_aggro_annotators)

#print the list length and a sample, for sanity-checking purposes
print(len(all_annotators))
print(all_annotators[0])

5781
{'worker_id': '85', 'gender': 'female', 'english_first_language': '0', 'age_group': '18-30', 'education': 'bachelors', 'dataset': 'toxicity', 'unique_id': '1'}


It worked! Now we can FINALLY start to analyze this dataset, and find out more about the workers who labelled the Wikipedia Talk Corpus.

## Part 2: Analyzing worker demographics

Now that we have our worker demographic data de-duplicated, combined and uniquely identified, we can start using it to ask and answer research questions! 

You are welcome to do this next steps in Python, here in this notebook. But if you aren't super comfortable with Python, you can run the cell below to export this dataset to a .csv file called ``a2_all_annotator_demographics.csv`` which you can open in Google Sheets and do the analysis there.

Note: if you choose Google Sheets, please title your Google Sheet "A2 worker demographics", share it with your instructor and TA ("can view" or "can edit") and paste a link to it in a Markdown cell below.

In [21]:
with open('a2_all_annotator_demographics.csv', 'w', encoding='utf-8') as f:
    writer = csv.writer(f)
    #write a header row
    writer.writerow(('unique_id',
                     'worker_id', 
                     'gender', 
                     'english_first_language', 
                     'age_group',
                     'education',
                     'dataset',))
    #loop through our dataset and write it to the file, row by row
    for a in all_annotators:
        writer.writerow((a['unique_id'], a['worker_id'], a['gender'], a['english_first_language'], a['age_group'], a['education'], a['dataset']))

### 2.1 Questions 1-4: compute descriptive statistics

Please create tables or bar charts that answer the following questions:

- **Q1.** What is the gender distribution of these workers?
- **Q2.** What is the distribution of the workers by first language?
- **Q3.** What is the distribution of these workers by age group?
- **Q4.** what is the distribution of these workers by education level?

If you *do not know* how to create tables in Python, you can use Excel or another tool and upload it to your notebook server. However I would encourage you to try out the [Pandas](https://pandas.pydata.org/pandas-docs/version/0.15/10min.html) library, which makes it very easy to convert your data into tidy tabular representations. If you do not know how to create charts in Python, you can use an easy library such as [Plotly](https://plotly.com/python/v3/statistics-charts/#new-to-plotly) to create charts.

In [22]:
# Read annotators .csv file into DataFrame
import pandas as pd
df_no_dupes = pd.read_csv("a2_all_annotator_demographics.csv")

# For Q1-4, read counts of each unique value for gender, english_first_language, age group, 
# and education level, then calculate percentages of each, rounding to 2 decimals, storing in 
# a unique DataFrame. 

# Q1: distribution of gender 

def getGenderDist(df):
    gender_dist = pd.DataFrame({
        "count": df["gender"].value_counts().astype(int),
        "percentage": (df["gender"].value_counts(normalize=True) * 100).round(2)
    })
    # adds a total row, summing up percentages to make sure they addd to 100
    gender_dist.loc["Total"] = [gender_dist["count"].sum(), "--"]
    return (gender_dist);

# Q2: distribution by english first language 
def getLanguageDist(df):
    english_dist = pd.DataFrame({
        "count": df["english_first_language"].value_counts().astype(int),
        "percentage": (df["english_first_language"].value_counts(normalize=True) * 100).round(2)
    })
    english_dist = english_dist.rename(
        index={0: "False", 1: "True"}
    )
    english_dist.loc["Total"] = [english_dist["count"].sum(), "--"]
    return english_dist;

# Q3: distribution of age
def getAgeDist(df):
    age_dist = pd.DataFrame({
        "count": df["age_group"].value_counts().astype(int),
        "percentage": (df["age_group"].value_counts(normalize=True) * 100).round(2)
    })
    age_dist.loc["Total"] = [age_dist["count"].sum(), "--"]
    return age_dist;

# Q4: distribution of education level
def getEducationDist(df):
    education_dist = pd.DataFrame({
        "count": df["education"].value_counts().astype(int),
        "percentage": (df["education"].value_counts(normalize=True) * 100).round(2)
    })
    education_dist.loc["Total"] = [education_dist["count"].sum(), "--"]
    return education_dist;

print(getGenderDist(df_no_dupes));
print();
print(getLanguageDist(df_no_dupes));
print();
print(getAgeDist(df_no_dupes));
print();
print(getEducationDist(df_no_dupes));
print();

        count percentage
gender                  
male     3676      63.59
female   2103      36.38
other       2       0.03
Total    5781         --

                        count percentage
english_first_language                  
False                    4713      81.53
True                     1068      18.47
Total                    5781         --

           count percentage
age_group                  
18-30       2911      51.35
30-45       2078      36.66
45-60        514       9.07
Under 18     116       2.05
Over 60       50       0.88
Total       5669         --

              count percentage
education                     
bachelors      2303      39.84
hs             1657      28.66
masters         931       16.1
professional    684      11.83
some            141       2.44
doctorate        61       1.06
none              4       0.07
Total          5781         --



- **Q1.** What is the gender distribution of these workers?

The workers are largely (64%) male. Still, more than a third (36%) are female. Those that identified as "other" are quite underrepresented, only two workers out of roughly 5000 (0.03%). 
- **Q2.** What is the distribution of the workers by first language?

Calling this a distribution of first language is a bit misleading, since the only data available is whether the worker's first language is English or not. That being said, most (82%) of workers did not have English as their first language, with the remaining (18%) with English as their first language. 
- **Q3.** What is the distribution of these workers by age group?

The age distribution is, unsurprisingly, skewed towards younger age groups, but not under 18. 

Over half (51%) of the workers are aged 18-30, about a third (37%) between 30 and 45. 

Other age groups make up 11% of the remaining workers, with 9% between 45 and 60, 2% under 18, and less than a percent over 60. 

- **Q4.** what is the distribution of these workers by education level?

The largest education group is bachelors' (just under 40%), which makes sense with the age distribution, after this, high school education is the second most popular group (29%), followed by masters (16%) and professional (12%). 

Other education groups make up less than 5% of the worker sample, with 2.5% having some education (presumably some college education, though this label is unclear), 1% with a doctorate, and very few (only 4 out of roughly 5000 workers, 0.07%) with no education. 

The "none" education label is also misleading, since it equates those just without a high school diploma and those who have never been schooled. 

### 2.2 Questions 5-8

Each of the next four questions requires a different kinds of answer.

Questions 5 and 6 have a "correct" answer, but we'll still give you credit for the "incorrect" answer as long as you document your process well (in Markdown, in this notebook). These questions are intended to get you thinking about how even small errors in data processing can create bias in your data.

Question 6 requires you to re-run some (but not all) of the data processing steps you performed above. Do this by copying the code into NEW cells below this line, and make sure to document each step using Markdown cells or inline (#) comments. You will probably want to change the names of the variables too.

- **Q5.** Analyze all_annotators in Sheets or Jupyter: Do any of the fields in this dataset have missing data? If so, which fields, and what % of the rows contain missing values?
- **Q6.** Build a version of the all_annotators dataset *without removing duplicates*, then re-run the summary statistics from questions 1-4. How have these summary statistics changed?

Questions 7 and 8 don't have one "correct" answer. These questions are intended to get you thinking about how using this dataset to train a machine learning model might lead to bias in the way that model performs when used as intended. You will need to provide a written answer to each of these questions in 3-4 full sentences. Consider what you know about the workers themselves, the task they were given, and about the purpose of the Perspective API.

Question 7 requires you to do some research of your own (hint: use Google). Question 8 doesn't require any new analysis or research, just critical thinking.

- **Q7.** In general, how do the demographics of these workers compare to those of English-speaking internet users overall? Why would it matter if the worker demographics don't match the demographics of the intended end users of the Perspective API?

- **Q8.** Given what you now know about the Wikipedia Talk Corpus, what issues might arise if it was used to train a machine learning-driven "hostile speech detector" that could be used on any website or social media platform?


**POST YOUR ANSWERS TO QUESTIONS 5-8 HERE.**

Use both Markdown and code-formatted cells.

In [62]:
# Q5: Do any field have missing data?

# Checks if all_annotators has missing fields
def isMissingValues(df):
    return df.isnull().values.any()

# Finds which fields have missing data and what % of rows have missing fields
def findNumMissingValues(df):
    missingRow = 0
    for column in df.columns:
        if (df[column].isnull().sum() != 0):
            missingRow += df[column].isnull().sum()

    return missingRow
    # (missingRow/(len(df_no_dupes.index)) * 100).round(2))

if isMissingValues(df_no_dupes):
    print("all_annotators has missing fields")

    numMissingValues = findNumMissingValues(df_no_dupes)
    print()
    print("all_annotators has ", numMissingValues, "missing values")
    print("Percentage of rows missing fields: ", (numMissingValues/(len(df_no_dupes.index)) * 100).round(2))

all_annotators has missing fields

all_annotators has  112 missing values
Percentage of rows missing fields:  1.94


**Q5.** : There are 112 missing fields in the age_group category, making up just under 2% of all rows. 

In [26]:
# Q6: Creating a dataset without removing duplicates and running Q1-Q4

# Make copies of the dictionaries in each dataset
toxicity_copy = [dict(w) for w in toxicity_annotators]
attack_copy = [dict(w) for w in attack_annotators]
aggression_copy = [dict(w) for w in aggression_annotators]

# Combine the copies while preserving all rows
toxicity_attack = finalize_dataset(toxicity_copy,attack_copy)
all_annotators_with_dupes = finalize_dataset(toxicity_attack, aggression_copy)

print("Total rows:", len(all_annotators_with_dupes))
print(all_annotators_with_dupes[0])

df_with_dupes = pd.DataFrame(all_annotators_with_dupes)

print(getGenderDist(df_with_dupes));
print();
print(getLanguageDist(df_with_dupes));
print();
print(getAgeDist(df_with_dupes));
print();
print(getEducationDist(df_with_dupes));
print();

Total rows: 7971
{'worker_id': '85', 'gender': 'female', 'english_first_language': '0', 'age_group': '18-30', 'education': 'bachelors', 'dataset': 'toxicity', 'unique_id': '1'}
        count percentage
gender                  
male     5025      63.04
female   2943      36.92
other       3       0.04
Total    7971         --

                        count percentage
english_first_language                  
0                        6501      81.56
1                        1470      18.44
Total                    7971         --

           count percentage
age_group                  
18-30       3960      49.68
30-45       2909      36.49
45-60        732       9.18
Under 18     153       1.92
             147       1.84
Over 60       70       0.88
Total       7971         --

              count percentage
education                     
bachelors      3165      39.71
hs             2288       28.7
masters        1316      16.51
professional    927      11.63
some            189       2

**Q6.** After running the demographic statistics from Q1-4 with duplicate data, the results don't change significantly. Each statistic is within about a percentage point of the analysis without duplicate data, which is surprising. Since adding duplicate data gives the duplicated data extra weight in demographic distributions, I would expect the reran statistics to be noticeably different. This might be explained if the demographics across all three files are similar to each other, creating a summary of the demographics across all three wouldn't be significantly different from demographic analyses across each. 

Let's check and see if this is the case. 

In [27]:
df_toxicity = pd.DataFrame(toxicity_copy)
# since aggression and attack contain the same data, we only have to do this once. 
df_aggression = pd.DataFrame(aggression_copy)

print(getGenderDist(df_toxicity));
print();
print(getGenderDist(df_aggression));
print();

print(getLanguageDist(df_toxicity));
print();
print(getLanguageDist(df_aggression));
print();

print(getAgeDist(df_toxicity));
print();
print(getAgeDist(df_aggression));
print();

print(getEducationDist(df_toxicity));
print();
print(getEducationDist(df_aggression));
print();

        count percentage
gender                  
male     2327       64.8
female   1263      35.17
other       1       0.03
Total    3591         --

        count percentage
gender                  
male     1349       61.6
female    840      38.36
other       1       0.05
Total    2190         --

                        count percentage
english_first_language                  
0                        2925      81.45
1                         666      18.55
Total                    3591         --

                        count percentage
english_first_language                  
0                        1788      81.64
1                         402      18.36
Total                    2190         --

           count percentage
age_group                  
18-30       1862      51.85
30-45       1247      34.73
45-60        296       8.24
Under 18      79        2.2
              77       2.14
Over 60       30       0.84
Total       3591         --

           count percentage
age_g

Great! Since each file contains demographic data that are similar enough (within a few percentage points) to each other, running demographic statistics with duplicate data doesn't skew the statistics too much. However, it's still important to remove duplicate data wherever it exists. 

**Q7.** 
Based on information collected from in 2021 by the US Census Bureau [1] and information from Statistica [2] on internet usage in the US and UK, the demographics of these workers have significant differences to those of English-speaking internet users overall. (Note: this analysis makes the assumption that most internet users in the UK and US are English-speaking and excludes other global English-speaking internet users. For the purposes of this general analysis, these assumptions are acceptable, but should be taken into account for a more in-depth analysis)

In both the US and UK, use of internet by gender is 50-50, which differs from the worker demographics of 64-36 for men and women, respectively. Age group differs as well, with populations older than 30 and younger than 18 making up a bigger share of internet users. Over 90% of each age group are regular internet users in the US. In the UK, age groups 20-29, 30-39, 40-49, and 50-59 all make up around 20% of total internet users. [2]

There are also differences in education, though this categroy is harder to analyze due to differences in education categories. Those with Bachelors' degrees in the UK only make up a smaller population (25% compared with 40%) than the worker demographics, and a greater percentage of people with high school education or less make up internet users in the US than the worker demographics (93% of this population are regular internet users) [2]. 

Lastly, language demographics might be the hardest to make assumptions about, but there still exist general differences between the worker data and English-speaking internet users overall. In the US, 90% of households with English as a secondary langauge are regular internet users [1]. Addtionally, a 2026 estimate shows that about three quaters of English speakers globally are nonnative speakers and 26% of internet users use English predominately [3]. All of this suggests that there might be a greater portion of global English-speaking internet users than the worker demographic percentage of 20%. 

[1] https://www2.census.gov/library/publications/2024/demo/acs-56.pdf 

[2] https://www.statista.com/topics/2237/internet-usage-in-the-united-states/#topicOverview 

[3] https://www.quantumrun.com/consulting/english-language/

**Q8.**
Given what you now know about the Wikipedia Talk Corpus, what issues might arise if it was used to train a machine learning-driven "hostile speech detector" that could be used on any website or social media platform?

The largest concern about training a machine learning-driven "hostile speech detetor" designed for any website or social media platform using only the Wikipedia Talk Corpus follows from the above analysis. The context of messages and demographics of internet users differs accross the internet and it is naive to assume a detetor trained for only one social context will be accurate accross different social contexts. 

The Center for Democracy and Technology published a report in 2017 detailing the limitations of automatic speech classifiers. The report identifies Domain-Specific tools (like the theoretical detator only trained within the domain of Wikipedia Talks Pages) as accurate within their speicifc domain, but notably inaccurate when applied to other domains. A detetor only trained on the Wikiepdia Talks Pages would be accurate when deteting toxic comments within the talks pages but would decrease in accuracy in any other context. Since there are significant demographic differences between the Corpus and broader English-speaking internet users, I would expect decreasing accuracy when processing comments from underrepresented demographics, such as individuals above 30 and below 18, women, those with below college education, and those with English as a nonnative language [4]. 

The emergence of false positives (i.e. flagging a comment as toxic when it was completely appropriate) was identificed as a common error with automatic speech classifiers. This would be a common issue if the "hostile speech detetor" was used on any other website or social media platform that it was not specifcally trained for. The report also documented gender and race bias of automatic classifiers. No race demographic data was given on the comment labelers, but given that more than half of the workers are men, it is likely that the "hostile speech detetor" would exhibit gender bias, risking the marginalization of other women and non-binary individuals through its use. The lack of race data for workers might suggest the same for marginalized racial groups [4]. 

[4] https://cdt.org/insights/mixed-messages-the-limits-of-automated-social-media-content-analysis/

## Part 3: Digging deeper

### 3.1 Questions 9-16

Below is a list of additional questions that you should be able to answer, based on what you've done today. 

***You may answer EITHER the "code" questions (Q9-Q12) OR the "no code" questions (Q13-Q16).***

If you don't feel very comfortable with Python yet, choose the "no code" questions.

- ***For the "code" questions*** you will need to load ``toxicity_annotations.tsv`` into Python and join that dataset with ``all_annotators`` on ``worker_id``.
- ***For the "no code" questions*** you will need to load the file ``toxicity_labelled_comments_3k_sample.csv`` into Google Sheets. This file contains the text of 3000 comments that were annotated for toxicity, as well as the toxicity score that each worker gave to these comments. The file is a random sample taken from the ~1.5 million comments contained in ``toxicity_annotated_comments.tsv``

*Note:* answering question 13-16 below ***will require reading some comments that contain offensive speech!*** If you do not want to be exposed to offensive speech, either answer the "code" questions instead, or reach out to your Instructor or TA and ask for an alternate activity.

#### Code questions
- **Q9.** What % of *workers* who labelled the toxicity dataset do we have demographic data for?
- **Q10.** What % of the *comments* in the toxicity dataset were labelled by male vs. female-identified workers?
- **Q11.** What % of the *comments* in the toxicity dataset were labelled by people for whom English is NOT their first language?
- **Q12.** Based on your findings from questions 9-11, how might this dataset present a biased view of "toxicity"? How would you expect such biases to impact how the Perspective API performs? 

#### No-code questions

Before you answer the questions below, read through at least 20 comments from ``toxicity_labelled_comments_3k_sample.csv`` that were labelled "toxic" (-1 or -2), 20 that were labelled "non-toxic" (1 or 2), and 20 that are labelled "neutral" (0). Note in your spreadsheet whether you agree or disagree with the labeller's judgement. 


- **Q13.** Pick 2-3 examples of comments where you disagreed with the labeller about the toxicity of a comment. Why did you disagree? Why do you think that the labeller might have labelled these comments the way they did?
- **Q14.** Pick 2-3 examples of comments where you don't understand what the commenter was saying, and therefore had a hard time classifying as toxic or non-toxic. What additional context or information would you need in order to be confident in your judgement about the toxicity of this comment?
- **Q15.** Read through the [instructions and labelling options](https://github.com/ewulczyn/wiki-detox/blob/master/src/modeling/toxicity_question.png) given to the crowdworkers. How could the way these instructions were written have made it difficult for crowdworkers to accurately label comments as toxic or non-toxic? If you were going to run a labelling campaign like this one yourself, how would you change these instructions or labelling options to help the crowdworkers make more accurate or consistent judgements about toxicity?
- **Q16.** Based on your findings from questions 13-15, how might this dataset present a biased view of "toxicity"? How would you expect such biases to impact how the Perspective API performs as a general-purpose hostile speech detector? 

**POST YOUR ANSWERS TO QUESTIONS 9-16 HERE.**

Use both Markdown and code-formatted cells.

In [54]:
# Q9-12

# Merge toxicity_annotations.tsv with all_annotators on worker_id
import pandas as pd

toxicity_annotations = pd.read_csv("toxicity_annotations.tsv", sep="\t")
toxicity_df = pd.DataFrame(toxicity_annotators)
toxicity_annotations["worker_id"] = (toxicity_annotations["worker_id"].astype(str))
joined_df = toxicity_annotations.merge(toxicity_df, on="worker_id", how="left")

print(joined_df.head())
print(joined_df.shape)

   rev_id worker_id  toxicity  toxicity_score  gender english_first_language  \
0  2232.0       723         0             0.0  female                      0   
1  2232.0      4000         0             0.0    male                      1   
2  2232.0      3989         0             1.0  female                      0   
3  2232.0      3341         0             0.0     NaN                    NaN   
4  2232.0      1574         0             1.0  female                      0   

  age_group  education   dataset unique_id  
0     30-45  bachelors  toxicity      1058  
1     18-30         hs  toxicity      3104  
2     18-30  bachelors  toxicity      2960  
3       NaN        NaN       NaN       NaN  
4     30-45  bachelors  toxicity       544  
(1598289, 10)


In [57]:
# Check the number of annotations before and after joining
print("Annotations before join:", len(toxicity_annotations))
print("Rows after join:", len(joined_df))
# Check for duplicate ids
print("Duplicate demographic worker IDs:", toxicity_df["worker_id"].duplicated().sum()

Annotations before join: 1598289
Rows after join: 1598289
251334
Duplicate demographic worker IDs: 0


In [64]:
# Q9: What % of workers do we have demographic data for

# Get the unique worker IDs from the annotations
annotation_workers = set(toxicity_annotations["worker_id"])
# Get the unique worker IDs from the demographic data
demographic_workers = set(toxicity_df["worker_id"])

# Find workers who have annotations but no demographic record
missing_workers = annotation_workers - demographic_workers

# Calculate the percentage
total_workers = len(annotation_workers)
num_missing = len(missing_workers)
missing_percentage = (num_missing / total_workers) * 100

print("Total workers:", total_workers)
print("Workers missing demographics:", num_missing)
print("Percentage missing demographics:", round(missing_percentage, 2), "%")

Total workers: 4301
Workers missing demographics: 710
Percentage missing demographics: 16.51 %


In [65]:
# Q10: What % of the comments in the toxicity dataset were labelled by male vs. female-identified workers?

# Count annotations by worker gender
gender_counts = joined_df["gender"].value_counts()

# Calculate percentages
gender_percentages = (joined_df["gender"].value_counts(normalize=True) * 100).round(2)

# Combine into a DataFrame
gender_dist = pd.DataFrame({"count": gender_counts, "percentage": gender_percentages})

print(gender_dist)

         count  percentage
gender                    
male    889251       66.02
female  457346       33.95
other      358        0.03


In [66]:
# Q11: What % of the comments in the toxicity dataset were labelled by people for whom English is 
# NOT their first language?

# Count annotations by whether English is the worker's first language
language_counts = joined_df["english_first_language"].value_counts()

# Calculate percentages
language_percentages = (joined_df["english_first_language"].value_counts(normalize=True) * 100).round(2)

# Combine into a DataFrame
language_dist = pd.DataFrame({"count": language_counts, "percentage": language_percentages})

print(language_dist)

                          count  percentage
english_first_language                     
0                       1113162       82.64
1                        233793       17.36


**Q12.** Based on your findings from questions 9-11, how might this dataset present a biased view of "toxicity"? How would you expect such biases to impact how the Perspective API performs?

Based on the data from Q9-11, it is clear that the workers annotating comments in the toxicity dataset are majority male and a larger majority with English as a first language. There are about 16% of toxicity annotators that we do not have demographic data for, though it is unlikely that the addition of demographic data for these annotators would skew the majority in favor of female or nonnative English speakers. 

Given the demographics of the annotators, this dataset has the potential to provide a misogynistic view of toxicity. It is possible that a large portion of the male annotators (who label the majority of comments) might be categorizing subtle or even overt misogynistic comments as less toxic than a female annotator would. A male-dominated annotator pool may underrepresent the perspectives of people who experience misogyny, namely women, potentially leading to some misogynistic comments being labeled as less toxic than they would be by a more representative group. If this bias were to be incorporated into the Perspective API, the tool would be much less reliable at classifying misogynistic comments than if it were trained with more female workers. 

Additionally, this dataset has the potential to provide a linguistically biased view of toxicity. Because more than 80% of comments were labelled by annotators with English as their first language, the dataset may not adequately represent the perspectives of people with different linguistic backgrounds. Expressions that are ordinary in a nonnative linguistic context may be interpreted differently by annotators unfamiliar with that context. Similarly, grammatical differences or unfamiliar phrasing could potentially be misinterpreted as hostility or toxicity. If these differences systematically affected annotation labels, the Perspective API might incorrectly classify some non-toxic comments as toxic, particularly in contexts involving diverse language backgrounds. 

Further, the annotator demographics dataset does not include information about the dialects of English spoken by annotators, nor does the demographic information alone establish which dialects are represented in the comments. The Center for Democracy and Technology report notes that automated language classifiers can over-flag neutral language written in African American Vernacular English (AAVE) as toxic or even as non-English [4]. If AAVE is underrepresented in the training data or its use is disproportionately associated with toxic labels, the Perspective API could reproduce or amplify this bias. Although our data cannot confirm whether AAVE speakers or their language are underrepresented, this remains a potential source of bias that warrants further investigation.

### 3.2 Challenge questions (optional)

The questions below are optional; you don't need to answer them to receive full credit for this assignment.

If you'd like to explore additional analyses of this dataset, here are a few questions to get you started!

- ***Challenge #1.*** Are female-identified workers more likely to label a comment as "toxic" than male identified workers? 
- ***Challenge #2.*** Are workers with a higher level of education more consistent in their "toxicity" labelling--in other words, do they tend to agree with other labellers more often? (remember that, according to the documentation, every comment was labelled by at least 10 crowdworkers). 
- ***Challenge #3.*** What are the words most frequently associated with toxic comments? (perhaps focus on comments where most or all of the workers agree are toxic)
- ***Challenge #4.*** What are the most polarizing comments--comments that some workers consider toxic, and others consider non-toxic? What about these comments made them so polarizing, or hard to classify?
- ***Challenge #5.*** Answer some or all of the homework questions above for one the other datasets ("attack" and "aggressive"). How are the potential sources of bias for these datasets the same, or different, than for the toxicity dataset? Would you "trust" a machine learning model based on these datasets more or less than one trained on the "toxicity" dataset? Why?

*OPTIONAL: POST YOUR ANSWERS TO THE CHALLENGE QUESTIONS HERE.*

Use both Markdown and code-formatted cells.